# Day 21 — HuggingFace: **Embeddings Pipeline**

**Phase 1 · Module 3: Prompt Engineering & RAG** · ~30 min

Yesterday you built cosine similarity by hand over toy vectors. Today you generate **real** embeddings the way production does — with a HuggingFace sentence-transformer — and store them in a **FAISS** index so top-*k* search stays fast when the corpus grows to millions of chunks.

Four steps:
1. Load `sentence-transformers/all-MiniLM-L6-v2` — the RAG workhorse.
2. Batch-encode 50 banking chunks with `show_progress_bar=True`.
3. Build a FAISS `IndexFlatL2` and query top-5.
4. `float32` vs `float64` — why embedding memory matters at scale.

> Heavy libs (`sentence-transformers`, `faiss`) are optional here — the notebook falls back to a pure-NumPy embedder + index so it **runs anywhere, no downloads, no keys**. The API you write is identical to production.

## 1. What a sentence-transformer is

`all-MiniLM-L6-v2` is a 6-layer distilled BERT that maps any sentence to a **384-dimension** vector via mean-pooling over token embeddings. It's the default RAG embedder because it's small (~80 MB), fast on CPU, and normalised so cosine ≈ dot product.

The contract is one method: `model.encode(list_of_texts) -> np.ndarray of shape (n, 384)`. Everything downstream — FAISS, the Bedrock Knowledge Base, reranking — consumes that matrix.

In [1]:
import numpy as np

try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    DIM = model.get_sentence_embedding_dimension()
    def encode(texts, **kw):
        return model.encode(texts, **kw).astype(np.float32)
    BACKEND = "sentence-transformers/all-MiniLM-L6-v2 (real)"
except Exception:
    # Keyless fallback: hashed bag-of-words -> fixed-dim L2-normalised vector.
    import re
    DIM = 384
    def encode(texts, show_progress_bar=False, batch_size=32):
        if isinstance(texts, str): texts = [texts]
        out = np.zeros((len(texts), DIM), dtype=np.float32)
        for r, t in enumerate(texts):
            for w in re.findall(r"[a-z0-9]+", t.lower()):
                out[r, hash(w) % DIM] += 1.0
            n = np.linalg.norm(out[r])
            if n: out[r] /= n
        return out
    BACKEND = "numpy hashed-bow (fallback)"

print("backend :", BACKEND)
print("dim     :", DIM)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

backend : sentence-transformers/all-MiniLM-L6-v2 (real)
dim     : 384


/var/folders/jd/b91jfh0n5r90fqg_0_2t9xg00000gn/T/ipykernel_2489/3771080749.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = model.get_sentence_embedding_dimension()


☝ Whichever backend loads, `encode` returns a `(n, DIM)` `float32` matrix — the fallback fakes the *vectors* but keeps the exact **shape and call signature** of the real model, so the FAISS code below is unchanged. In production you install `sentence-transformers` and the real branch runs.

## 2. Batch-encode 50 chunks

Encoding one text at a time wastes the GPU/CPU — you pay per-call overhead 50 times. `encode` **batches** internally; pass the whole list and set `batch_size`. `show_progress_bar=True` prints a tqdm bar, which matters when you're embedding 100k chunks and want an ETA.

In [2]:
# 50 banking policy chunks (10 templates x 5 products)
products = ["residential mortgage", "buy-to-let mortgage", "personal loan", "overdraft", "credit card"]
templates = [
    "The {p} requires proof of income for approval.",
    "Early repayment of a {p} may incur a fee.",
    "The maximum term for a {p} is twenty-five years.",
    "Missed payments on a {p} are reported to credit agencies.",
    "A {p} is subject to affordability assessment.",
    "Interest on a {p} is charged monthly.",
    "The {p} has a minimum age requirement of eighteen.",
    "Applicants for a {p} must be UK residents.",
    "The {p} annual percentage rate is fixed at origination.",
    "Complaints about a {p} follow the FCA process.",
]
chunks = [t.format(p=p) for p in products for t in templates]
print("chunks:", len(chunks))

emb = encode(chunks, show_progress_bar=True, batch_size=16)
print("embeddings shape:", emb.shape, "dtype:", emb.dtype)

chunks: 50


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

embeddings shape: (50, 384) dtype: float32


☝ One `encode` call embedded all 50 chunks and returned a `(50, 384)` matrix. Batching means the model runs ~4 forward passes (`batch_size=16`), not 50 — the single biggest ingestion speed-up you get for free.

## 3. FAISS `IndexFlatL2` — top-*k* search

A brute-force NumPy scan works for 50 vectors; at a million it's too slow. **FAISS** is the standard vector index. `IndexFlatL2` is the simplest: it stores every vector and does exact **L2 (Euclidean)** nearest-neighbour search — the baseline you build before reaching for approximate indexes (IVF, HNSW).

For **L2-normalised** vectors (what MiniLM returns), smaller L2 distance ⇔ larger cosine, so ranking is identical to yesterday's cosine — FAISS just does it faster.

In [3]:
try:
    import faiss
    index = faiss.IndexFlatL2(DIM)
    index.add(emb)
    def search(qvec, k):
        D, I = index.search(qvec, k)
        return list(zip(I[0].tolist(), D[0].tolist()))
    IDX = "faiss.IndexFlatL2 (real)"
except Exception:
    class FlatL2:                       # numpy stand-in, same L2 semantics
        def __init__(self, d): self.v = np.empty((0, d), np.float32)
        def add(self, x): self.v = np.vstack([self.v, x])
        def search(self, q, k):
            d = ((self.v - q) ** 2).sum(1)
            i = np.argsort(d)[:k]
            return d[i][None, :], i[None, :]
    index = FlatL2(DIM); index.add(emb)
    def search(qvec, k):
        D, I = index.search(qvec, k)
        return list(zip(I[0].tolist(), D[0].tolist()))
    IDX = "numpy FlatL2 (fallback)"

print("index :", IDX, "| vectors:", len(chunks))

query = "what fee applies if I pay off my mortgage early"
qv = encode([query])
print("\nquery:", query)
for rank, (i, dist) in enumerate(search(qv, 5), 1):
    print(f"  {rank}. L2={dist:6.3f}  {chunks[i]}")

index : faiss.IndexFlatL2 (real) | vectors: 50

query: what fee applies if I pay off my mortgage early
  1. L2= 0.536  Early repayment of a residential mortgage may incur a fee.
  2. L2= 0.668  Early repayment of a buy-to-let mortgage may incur a fee.
  3. L2= 0.853  Early repayment of a personal loan may incur a fee.
  4. L2= 0.872  Early repayment of a credit card may incur a fee.
  5. L2= 0.910  Early repayment of a overdraft may incur a fee.


☝ `index.add(emb)` loads the matrix once; `index.search(qv, 5)` returns the 5 nearest chunk ids and their L2 distances. The early-repayment chunk ranks top — the same result yesterday's cosine gave, but now served by an index that scales to millions of vectors with `IndexIVFFlat` swapped in for one line.

## 4. `float32` vs `float64` — memory at scale

Embeddings default to `float64` in NumPy if you're not careful — **8 bytes** per number. Real models and FAISS use `float32` (**4 bytes**), halving RAM at no measurable recall cost. At Barclays corpus scale the difference is gigabytes.

In [4]:
n, d = 5_000_000, DIM        # 5M chunks at this dimension
for dt in (np.float64, np.float32):
    gb = n * d * np.dtype(dt).itemsize / 1e9
    print(f"{np.dtype(dt).name}: {np.dtype(dt).itemsize} bytes/val  ->  {gb:6.1f} GB for {n:,} x {d}")

print("\nour matrix dtype:", emb.dtype, "( float32 -> half the RAM of float64 )")

float64: 8 bytes/val  ->    15.4 GB for 5,000,000 x 384
float32: 4 bytes/val  ->     7.7 GB for 5,000,000 x 384

our matrix dtype: float32 ( float32 -> half the RAM of float64 )


☝ Same vectors, half the memory — that's why every embedding pipeline casts to `float32` (note the `.astype(np.float32)` in step 1). `float16` halves it again for storage, but keep search in `float32` to avoid precision loss in the distance sum.

## 5. Recap — the ingestion pipeline

| Step | Call | Why |
|---|---|---|
| **Load model** | `SentenceTransformer("all-MiniLM-L6-v2")` | 384-dim, CPU-fast, RAG default |
| **Batch encode** | `encode(chunks, batch_size, show_progress_bar)` | one pass over many chunks, not one-per-call |
| **Index** | `faiss.IndexFlatL2(dim).add(emb)` | exact NN baseline; swap IVF/HNSW to scale |
| **Query** | `index.search(encode([q]), k)` | top-*k* by L2 ⇔ cosine on normalised vecs |
| **`float32`** | `.astype(np.float32)` | half the RAM, no recall cost |

